# syft-ingest API Exploration

In [1]:
import syft_ingest as si
from syft_ingest import async_gather, gather

## 1. Simplified gather() API — YouTube video

In [ ]:
# Simplest case: just platform + URLs
corpus = si.gather(
    "youtube", ["https://www.youtube.com/watch?v=zY2dAK-pMPI&t=11s"]
)  # Andrew Trask: talk

print(f"Items fetched: {len(corpus.all_items())}")
if corpus.all_items():
    item = corpus.all_items()[0]
    print(f"  Title: {item.title}")
    print(f"  Author: {item.author}")

In [ ]:
corpus.export("./data/youtube.jsonl")

## 2. Channel enumeration with config

In [ ]:
# With config options passed as kwargs
corpus = si.gather(
    "youtube",
    ["https://www.youtube.com/@iamtrask"],
    playlistend=3,  # Cap at 3 videos to keep it fast
)

print(f"Videos found: {len(corpus.all_items())}")
for item in corpus.all_items():
    print(f"  [{item.published_at}] {item.title}")

In [ ]:
corpus.export("./data/youtube2.jsonl")

## 3. Facebook scraping

Requires `BRIGHTDATA_API_TOKEN`. Triggers a real scrape job and polls until done (~60-120s).

In [2]:
import os

token = os.getenv("BRIGHTDATA_API_TOKEN")
print(
    f"BRIGHTDATA_API_TOKEN : {token[:3] + '...' + token[-3:] if token else 'NOT SET -- BrightData cells will fail'}"
)

BRIGHTDATA_API_TOKEN : 7a5...1d5


In [ ]:
corpus = await async_gather(
    "facebook",
    ["https://www.facebook.com/profile.php?id=61583734012155"],
    author="Syft Influencer Test",
)

print(f"Items fetched: {len(corpus.all_items())}")
print(f"Author: {corpus.person}")

if corpus.all_items():
    item = corpus.all_items()[0]
    print("\nFirst item:")
    print(f"  Type: {type(item).__name__}")
    print(f"  Title: {item.title}")
    print(f"  Text: {(item.text or '')[:120]}")

In [ ]:
corpus.export("data/fb.jsonl")

## 4. Instagram scraping with posts_limit for testing

In [ ]:
corpus = await async_gather(
    "instagram",
    ["https://www.instagram.com/syft_influencer_test/"],
    author="Syft Influencer Test",
    timeout=120,
    poll_interval=5,
)

print(f"Items fetched: {len(corpus.all_items())}")
for i, item in enumerate(corpus.all_items()[:3], 1):
    print(f"\n{i}. {type(item).__name__}: {item.title}")

2026-04-13 21:28:10.273 | DEBUG    | syft_ingest.sources.youtube:__init__:83 - YtDlpFetcher initialized with config: {'socket_timeout': 30, 'playlistend': 50, 'download_full_video': False}
2026-04-13 21:28:10.273 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher YtDlpFetcher for youtube/yt-dlp
2026-04-13 21:28:10.273 | DEBUG    | syft_ingest.setup:register_fetchers:28 - Registered YtDlpFetcher for YouTube
2026-04-13 21:28:10.273 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher BrightDataFetcher for facebook/brightdata
2026-04-13 21:28:10.274 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher BrightDataFetcher for instagram/brightdata
2026-04-13 21:28:10.274 | DEBUG    | syft_ingest.setup:register_fetchers:37 - Registered BrightDataFetcher for Facebook and Instagram
2026-04-13 21:28:10.274 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher LocalFetcher for local/local
2

FetchError: Unexpected error: InstagramScraper.profiles_trigger() got an unexpected keyword argument 'num_of_posts'

## 7. Error handling and validation

In [ ]:
# Invalid platform raises ValueError
try:
    corpus = gather("tiktok", ["https://tiktok.com/user"])
except KeyError as e:
    print(f"✅ ValueError caught for unsupported platform: {e}")

# Missing URLs raises ValueError
try:
    corpus = gather("youtube", None)
except ValueError as e:
    print(f"✅ ValueError caught for missing URLs: {e}")